# 인용 쌍으로 임베딩 파인튜닝 (Colab T4)

**이 노트북이 답하는 질문 두 개**

1. `bge-base-en-v1.5`는 `text-embedding-3-large`와 **동등합니다**(recall@10 0.1909 대 0.2019,
   95% CI [-0.0250, +0.0032]로 0 포함). 인용 쌍으로 파인튜닝하면 **넘는가?**
2. 그래프에서 뽑은 **hard negative**는 선형 어댑터에서 기각됐습니다(정규화 6개 수준 전부에서
   이기지 못함). 용량 있는 모델에서는 **작동하는가?**

두 질문 다 GPU 없이는 답할 수 없습니다. 개발 PC에는 NVIDIA가 없어(Intel Iris Xe) 37,854편
인코딩이 CPU로 3.5시간이고 파인튜닝은 비현실적입니다.

**범위.** 오프라인 측정까지입니다. 앱에 넣으려면 ① 37,854편 재임베딩 ② 질의 시 로컬 모델 추론이
필요하고, ②가 torch를 앱 의존성으로 끌어옵니다(현재 핵심 경로 259MB, Streamlit 보장 690MB).
배포는 별도 판단입니다.

**실행 순서.** `SMOKE = True`로 한 번 돌려 코드가 도는지 보고, 그다음 `False`로 본 실행을 하세요.
SMOKE 수치는 말뭉치를 줄여 재므로 **결과로 읽지 마세요.**

In [18]:
# 설정: SMOKE=True면 작은 부분만으로 코드가 도는지 확인합니다(CPU도 가능).
SMOKE = False

BRANCH = "main"
REPO_URL = "https://github.com/rndigkwk/arxiv_graph_RAG.git"

# HF 저장소 이름만 적습니다. 계정명은 토큰에서 꺼내므로 손으로 적지 않습니다.
# 다른 계정의 저장소를 쓸 때만 DATA_REPO_OWNER를 채우세요.
DATA_REPO_NAME = "arxiv-graphrag-colab"
DATA_REPO_OWNER = None

BASE_MODEL = "BAAI/bge-base-en-v1.5"

# 제목+초록 토큰 중앙값이 258입니다. 128이나 256으로 자르면 논문 절반이 잘려
# 모델이 아니라 절단을 재게 됩니다.
MAX_SEQ_LENGTH = 512

# 손실의 배치 크기(배치 안 음성의 품질을 정합니다)와 한 번에 GPU에 올리는 양을 나눕니다.
# 하드 네거티브가 있으면 예제당 텍스트가 3개이므로 32예제 = 96텍스트이고, 그것을 한 번에
# 올리면 T4 15GB에서 터집니다. CachedMultipleNegativesRankingLoss가 임베딩을 캐시해
# 손실 배치는 유지하면서 GPU에는 MINI_BATCH개 텍스트만 올립니다.
BATCH_SIZE = 32

# MINI_BATCH는 추측하지 않고 재서 정합니다. 32는 OOM이 났고 4는 15GB 중 1.87GB만 써서
# 87%를 놀렸습니다. 아래 셀이 후보를 몇 스텝씩 돌려 실제 최대 메모리를 보고 고릅니다.
AUTO_MINI_BATCH = True
MINI_BATCH_CANDIDATES = (8, 16, 32, 64, 96)
MINI_BATCH = 16          # AUTO_MINI_BATCH=False일 때 쓰는 값
MEMORY_HEADROOM = 0.75   # 전체 메모리의 이 비율까지만 쓰는 후보를 고릅니다

EVAL_BATCH = 16
LEARNING_RATE = 2e-5
EPOCHS = 1
SEED = 20261015
TOP_K = 10
HF_REPO_PREFIX = "bge-arxiv-citation"   # 계정명은 토큰에서 꺼내 앞에 붙습니다

SMOKE_CORPUS = 3000     # SMOKE에서 후보로 쓸 논문 수
SMOKE_PAIRS = 400
SMOKE_QUERIES = 40

In [19]:
import os
import sys

# 조각화로 인한 OOM을 줄입니다. torch를 가져오기 전에 설정해야 효과가 있습니다.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("python", sys.version.split()[0])
try:
    import torch

    print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"gpu {torch.cuda.get_device_name(0)} | {total:.1f} GiB")
except ImportError:
    print("torch 아직 없음 (다음 셀에서 설치)")

python 3.12.13
torch 2.11.0+cu128 | cuda True
gpu Tesla T4 | 14.6 GiB


In [20]:
# 저장소와 데이터를 받습니다. 로컬에서 SMOKE로 돌릴 때는 저장소 루트에서 실행되므로 clone을 건너뜁니다.
import os
from pathlib import Path

ON_COLAB = os.path.exists("/content")
if ON_COLAB:
    %cd /content
    !rm -rf arxiv_graph_RAG
    !git clone --depth 1 --branch {BRANCH} {REPO_URL}
    %cd /content/arxiv_graph_RAG
    !pip install -q "sentence-transformers[train]>=5.1" "datasets>=2.19"

# Hugging Face 토큰: VS Code Colab 확장에서는 Colab 비밀 값을 못 읽어 직접 입력합니다.
token = None
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    pass
if token is None:
    from getpass import getpass

    token = getpass("Hugging Face 토큰 (쓰기 권한): ")

# 계정명을 토큰에서 꺼냅니다. 자리표시자를 손으로 바꾸게 하면 빠뜨렸을 때
# huggingface_hub 안쪽에서 HFValidationError가 나와 원인을 찾기 어렵습니다.
from huggingface_hub import HfApi

HF_USER = HfApi(token=token).whoami()["name"]
DATA_REPO = f"{DATA_REPO_OWNER or HF_USER}/{DATA_REPO_NAME}"
print("HF 계정", HF_USER, "| 데이터 저장소", DATA_REPO)

DATA_DIR = Path("exports/colab")
if not (DATA_DIR / "manifest.json").exists():
    from huggingface_hub import snapshot_download

    snapshot_download(
        DATA_REPO, repo_type="dataset", token=token,
        local_dir=".", allow_patterns=["exports/colab/*"],
    )
if not (DATA_DIR / "manifest.json").exists():
    raise SystemExit(
        f"{DATA_REPO} 에서 exports/colab/manifest.json 을 받지 못했습니다. "
        "업로드할 때 저장소 안 경로가 exports/colab/... 인지 확인하세요: "
        "hf repo files <저장소> --repo-type dataset"
    )

import importlib.metadata as metadata

for package in ("sentence-transformers", "transformers", "torch", "datasets", "accelerate"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "없음")

/content
Cloning into 'arxiv_graph_RAG'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 243 (delta 46), reused 179 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (243/243), 124.35 MiB | 21.96 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Updating files: 100% (211/211), done.
/content/arxiv_graph_RAG
HF 계정 blanden77 | 데이터 저장소 blanden77/arxiv-graphrag-colab


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

sentence-transformers 5.6.0
transformers 5.13.1
torch 2.11.0+cu128
datasets 4.0.0
accelerate 1.14.0


In [21]:
import gzip
import json
import random

import numpy as np

random.seed(SEED)
np.random.seed(SEED)


def read_jsonl(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
corpus_rows = read_jsonl(DATA_DIR / "corpus.jsonl.gz")
train_rows = read_jsonl(DATA_DIR / "train_pairs.jsonl.gz")
eval_rows = read_jsonl(DATA_DIR / "eval_queries.jsonl.gz")
baseline_rows = read_jsonl(DATA_DIR / "baseline_recall.jsonl.gz")

print("snapshot", manifest["snapshot_version"], "| papers sha256",
      manifest["snapshot_papers_sha256"][:16])
print("corpus", len(corpus_rows), "| train pairs", len(train_rows),
      "| eval queries", len(eval_rows))
print("baseline", manifest["baseline"]["model"], "recall@10",
      manifest["baseline"]["recall_at_10"])
print(manifest["max_seq_length_note"])

snapshot 2026-10-14 | papers sha256 662ec63c7255286c
corpus 37854 | train pairs 36554 | eval queries 365
baseline text-embedding-3-large recall@10 0.2019
제목+초록 토큰 중앙값이 258입니다. 128이나 256으로 자르면 논문 절반이 잘려 모델이 아니라 절단을 재게 됩니다. 512를 쓰세요.


In [22]:
# SMOKE에서는 말뭉치·쌍·질의를 줄입니다. 후보 수가 달라지므로 SMOKE 수치는 비교에 쓸 수 없습니다.
if SMOKE:
    keep_ids = {row["anchor"] for row in eval_rows[:SMOKE_QUERIES]}
    for row in eval_rows[:SMOKE_QUERIES]:
        keep_ids.update(row["gold"])
    extra = [row["arxiv_id"] for row in corpus_rows if row["arxiv_id"] not in keep_ids]
    keep_ids.update(extra[: max(0, SMOKE_CORPUS - len(keep_ids))])

    corpus_rows = [row for row in corpus_rows if row["arxiv_id"] in keep_ids]
    eval_rows = eval_rows[:SMOKE_QUERIES]
    train_rows = [
        row for row in train_rows
        if {row["anchor"], row["positive"], row["hard_negative"]} <= keep_ids
    ][:SMOKE_PAIRS]
    print("SMOKE: corpus", len(corpus_rows), "pairs", len(train_rows),
          "queries", len(eval_rows))
    print("경고: 후보 수를 줄였으므로 아래 수치를 결과로 읽지 마세요.")

text_of = {row["arxiv_id"]: row["text"] for row in corpus_rows}
corpus_ids = [row["arxiv_id"] for row in corpus_rows]
position = {arxiv_id: index for index, arxiv_id in enumerate(corpus_ids)}
baseline_recall = {row["anchor"]: row["recall_at_10"] for row in baseline_rows}

queries = [
    (row["anchor"], [paper for paper in row["gold"] if paper in position])
    for row in eval_rows
    if row["anchor"] in position
]
queries = [(anchor, gold) for anchor, gold in queries if gold]
print("사용할 질의", len(queries))

사용할 질의 365


In [23]:
import torch
from sentence_transformers import SentenceTransformer


def free_gpu(*objects):
    """모델을 GPU에서 내리고 캐시를 비웁니다.

    모델 셋(튜닝 전 + 변형 둘)을 동시에 올려 두면 T4 15GB가 모자랍니다. 평가가 끝난 모델은
    바로 내려야 다음 학습이 들어갈 자리가 생깁니다.
    """
    for item in objects:
        try:
            item.to("cpu")
        except Exception:
            pass
    import gc

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_free_gib():
    if not torch.cuda.is_available():
        return None
    free, _ = torch.cuda.mem_get_info()
    return round(free / 1024**3, 2)


def encode_all(model, batch_size=None):
    vectors = model.encode(
        [text_of[arxiv_id] for arxiv_id in corpus_ids],
        batch_size=batch_size or EVAL_BATCH, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    )
    return np.asarray(vectors, dtype=np.float32)


def per_query_recall(vectors, top_k=TOP_K):
    """질의 논문 자신을 후보에서 빼고 전체 말뭉치를 순위 매깁니다."""
    rows = np.stack([vectors[position[anchor]] for anchor, _ in queries])
    similarity = rows @ vectors.T
    for index, (anchor, _) in enumerate(queries):
        similarity[index, position[anchor]] = -np.inf
    ranked = np.argpartition(-similarity, top_k, axis=1)[:, :top_k]
    recalls, hits, reciprocal = [], [], []
    for index, (_, gold) in enumerate(queries):
        order = ranked[index][np.argsort(-similarity[index, ranked[index]])]
        retrieved = [corpus_ids[spot] for spot in order]
        found = [rank for rank, paper in enumerate(retrieved, start=1) if paper in set(gold)]
        recalls.append(len(set(gold) & set(retrieved)) / len(gold))
        hits.append(float(bool(found)))
        reciprocal.append(1.0 / found[0] if found else 0.0)
    return np.asarray(recalls), np.asarray(hits), np.asarray(reciprocal)


def metrics(vectors):
    recalls, hits, reciprocal = per_query_recall(vectors)
    return {
        "queries": len(queries),
        f"recall_at_{TOP_K}": round(float(recalls.mean()), 4),
        f"hit_at_{TOP_K}": round(float(hits.mean()), 4),
        "mrr": round(float(reciprocal.mean()), 4),
    }, recalls


def paired_bootstrap(baseline, candidate, samples=2000):
    """질의를 쌍으로 재표본합니다. 평균이 질의 몇 건 차이일 때 그것이 측정인지 가릅니다."""
    difference = candidate - baseline
    generator = np.random.default_rng(SEED)
    indices = generator.integers(0, len(difference), size=(samples, len(difference)))
    means = difference[indices].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {
        "mean_difference": round(float(difference.mean()), 4),
        "ci95_low": round(float(low), 4),
        "ci95_high": round(float(high), 4),
        "excludes_zero": bool(low > 0.0 or high < 0.0),
    }


openai_recall = np.asarray([baseline_recall[anchor] for anchor, _ in queries])
print("OpenAI 기준선 recall@10 (이 질의 집합)", round(float(openai_recall.mean()), 4))

OpenAI 기준선 recall@10 (이 질의 집합) 0.2019


In [24]:
# 튜닝 전 기준선. SMOKE가 아니라면 recall@10이 0.1909 근처여야 합니다.
# 크게 다르면 토크나이저·정규화·질의 집합 중 하나가 어긋난 것이고, 그때는 비교가 성립하지 않습니다.
untuned = SentenceTransformer(BASE_MODEL)
untuned.max_seq_length = MAX_SEQ_LENGTH
untuned_metrics, untuned_recall = metrics(encode_all(untuned))
print("untuned", untuned_metrics)
if not SMOKE:
    gap = abs(untuned_metrics[f"recall_at_{TOP_K}"] - 0.1909)
    print("기대값 0.1909과의 차이", round(gap, 4),
          "-> 재현됨" if gap < 0.01 else "-> 어긋남, 비교 전에 원인을 찾으세요")

# 점수만 남기고 모델은 바로 내립니다. 셋을 동시에 올려 두면 T4 15GB가 모자랍니다.
free_gpu(untuned)
del untuned
print("GPU 여유", gpu_free_gib(), "GiB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2366 [00:00<?, ?it/s]

untuned {'queries': 365, 'recall_at_10': 0.1909, 'hit_at_10': 0.5671, 'mrr': 0.399}
기대값 0.1909과의 차이 0.0 -> 재현됨
GPU 여유 13.6 GiB


In [25]:
import csv

from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from transformers import TrainerCallback


class LossCsvCallback(TrainerCallback):
    """스텝별 손실을 CSV로 적습니다. `report_to=[]`라 다른 기록이 남지 않기 때문입니다.

    Colab에서 MLflow를 켜면 `mlruns/`를 그 VM에 쓰고 런타임이 끊기면 같이 사라집니다. 원격
    추적 서버가 없는 동안은 파일로 남겨 보고서와 함께 올리는 쪽이 실제로 보존됩니다. 결과
    지표만으로는 수렴했는지, 더 돌릴 가치가 있는지를 나중에 되짚을 수 없습니다.
    """

    FIELDS = ("variant", "step", "epoch", "loss", "learning_rate", "grad_norm")

    def __init__(self, variant, path):
        self.variant = variant
        self.path = path
        self.history = []
        with open(self.path, "w", encoding="utf-8", newline="") as handle:
            csv.writer(handle).writerow(self.FIELDS)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return
        row = {
            "variant": self.variant,
            "step": state.global_step,
            "epoch": round(float(logs.get("epoch") or 0.0), 4),
            "loss": float(logs["loss"]),
            "learning_rate": logs.get("learning_rate"),
            "grad_norm": logs.get("grad_norm"),
        }
        self.history.append(row)
        with open(self.path, "a", encoding="utf-8", newline="") as handle:
            csv.writer(handle).writerow([row[field] for field in self.FIELDS])

    def summary(self):
        """보고서에도 수렴을 읽을 수 있을 만큼은 넣습니다."""
        if not self.history:
            return {"logged_steps": 0}
        losses = [row["loss"] for row in self.history]
        return {
            "logged_steps": len(losses),
            "first": round(losses[0], 4),
            "last": round(losses[-1], 4),
            "min": round(min(losses), 4),
            "mean_last_quarter": round(
                sum(losses[-max(1, len(losses) // 4):]) / max(1, len(losses) // 4), 4
            ),
            "csv": self.path,
        }


def build_dataset(with_hard_negative):
    """하드 네거티브를 열로 주면 MNR이 그것을 배치 안 후보에 더합니다."""
    columns = {
        "anchor": [text_of[row["anchor"]] for row in train_rows],
        "positive": [text_of[row["positive"]] for row in train_rows],
    }
    if with_hard_negative:
        columns["negative"] = [text_of[row["hard_negative"]] for row in train_rows]
    return Dataset.from_dict(columns)


def train_and_score(variant, with_hard_negative):
    """한 변형을 학습하고 바로 채점한 뒤 GPU에서 내립니다.

    모델을 모아 두고 나중에 평가하면 T4에서 OOM이 납니다. 학습 -> 인코딩 -> 채점 -> 해제를
    변형마다 닫아서, 어느 순간에도 모델 하나만 GPU에 있게 합니다.
    """
    model = SentenceTransformer(BASE_MODEL)
    model.max_seq_length = MAX_SEQ_LENGTH
    arguments = SentenceTransformerTrainingArguments(
        output_dir=f"/tmp/out-{variant}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        logging_steps=50,
        save_strategy="no",
        report_to=[],
    )
    curve = LossCsvCallback(variant, f"/tmp/loss-{variant}.csv")
    trainer = SentenceTransformerTrainer(
        model=model,
        args=arguments,
        train_dataset=build_dataset(with_hard_negative),
        # 손실 배치는 BATCH_SIZE로 두고 GPU에는 MINI_BATCH만 올립니다. 배치 안 음성의
        # 품질을 지키면서 메모리를 묶는 방법이며, 기울기 누적으로는 대체할 수 없습니다.
        loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH),
        callbacks=[curve],
    )
    trainer.train()
    model.save(f"/tmp/out-{variant}")

    row, recall = metrics(encode_all(model))
    print(variant, row)
    print("  손실", curve.summary())
    free_gpu(model)
    del model, trainer
    print("  GPU 여유", gpu_free_gib(), "GiB")
    return row, recall, curve.summary()


scored = {}

In [26]:
def probe_mini_batch(candidates=MINI_BATCH_CANDIDATES, steps=3, headroom=MEMORY_HEADROOM):
    """몇 스텝씩 실제로 돌려 보고 들어가는 가장 큰 MINI_BATCH를 고릅니다.

    추측으로 두 번 틀렸습니다. 32는 OOM이 났고, 4로 내리니 15GB 중 1.87GB만 써서 87%를
    놀리고 그만큼 느렸습니다. GradCache의 메모리는 이 값에 거의 비례하므로, 재서 고르면
    두 실패를 모두 피할 수 있습니다.

    하드 네거티브가 있는 쪽으로 재는 이유는 그쪽이 예제당 텍스트가 3개라 더 무겁기
    때문입니다. 무거운 쪽에서 들어가면 가벼운 쪽도 들어갑니다.
    """
    if not torch.cuda.is_available():
        print("GPU 없음 -> MINI_BATCH", MINI_BATCH, "그대로 사용")
        return MINI_BATCH

    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    probe_rows = train_rows[: max(BATCH_SIZE * (steps + 1), BATCH_SIZE * 2)]
    dataset = Dataset.from_dict({
        "anchor": [text_of[row["anchor"]] for row in probe_rows],
        "positive": [text_of[row["positive"]] for row in probe_rows],
        "negative": [text_of[row["hard_negative"]] for row in probe_rows],
    })

    chosen, measurements = None, []
    for candidate in candidates:
        model = SentenceTransformer(BASE_MODEL)
        model.max_seq_length = MAX_SEQ_LENGTH
        torch.cuda.reset_peak_memory_stats()
        try:
            trainer = SentenceTransformerTrainer(
                model=model,
                args=SentenceTransformerTrainingArguments(
                    output_dir=f"/tmp/probe-{candidate}",
                    max_steps=steps,
                    per_device_train_batch_size=BATCH_SIZE,
                    learning_rate=LEARNING_RATE,
                    fp16=True,
                    seed=SEED,
                    logging_strategy="no",
                    save_strategy="no",
                    report_to=[],
                ),
                train_dataset=dataset,
                loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=candidate),
            )
            trainer.train()
            peak = torch.cuda.max_memory_allocated() / 1024**3
            fits = peak <= total * headroom
            measurements.append((candidate, round(peak, 2), fits))
            print(f"  mini_batch {candidate:3d} -> 최대 {peak:.2f} GiB / {total:.1f} GiB"
                  f" {'(여유)' if fits else '(한도 초과)'}")
            if fits:
                chosen = candidate
            free_gpu(model)
            del model, trainer
            if not fits:
                break
        except torch.cuda.OutOfMemoryError:
            measurements.append((candidate, None, False))
            print(f"  mini_batch {candidate:3d} -> OOM")
            free_gpu(model)
            del model
            break

    print("고른 MINI_BATCH:", chosen or candidates[0])
    return chosen or candidates[0]


if AUTO_MINI_BATCH and not SMOKE:
    MINI_BATCH = probe_mini_batch()
print("MINI_BATCH =", MINI_BATCH, "| GPU 여유", gpu_free_gib(), "GiB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


  mini_batch   8 -> 최대 3.71 GiB / 14.6 GiB (여유)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

  mini_batch  64 -> 최대 8.17 GiB / 14.6 GiB (여유)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


  mini_batch  96 -> 최대 8.17 GiB / 14.6 GiB (여유)
고른 MINI_BATCH: 96
MINI_BATCH = 96 | GPU 여유 12.6 GiB


In [ ]:
scored["inbatch"] = train_and_score("inbatch", with_hard_negative=False)
scored["hard"] = train_and_score("hard", with_hard_negative=True)
print("학습·채점 완료", list(scored))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,2.208734
100,1.870222
150,1.808420
200,1.759395
250,1.772583
300,1.583183
350,1.621973
400,1.569580
450,1.628623
500,1.603305


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2366 [00:00<?, ?it/s]

inbatch {'queries': 365, 'recall_at_10': 0.224, 'hit_at_10': 0.6411, 'mrr': 0.441}
  손실 {'logged_steps': 22, 'first': 2.2087, 'last': 1.5444, 'min': 1.4717, 'mean_last_quarter': 1.5109, 'csv': '/tmp/loss-inbatch.csv'}
  GPU 여유 12.62 GiB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
50,2.776373
100,2.382173


In [ ]:
results = {
    "text-embedding-3-large": {
        "recall_at_10": round(float(openai_recall.mean()), 4),
        "source": "스냅샷에서 내보낸 질의별 점수",
    },
    f"{BASE_MODEL} (untuned)": {
        **untuned_metrics,
        "vs_openai": paired_bootstrap(openai_recall, untuned_recall),
    },
}
for variant, (row, recall, curve) in scored.items():
    results[f"{BASE_MODEL} ({variant})"] = {
        **row,
        "vs_openai": paired_bootstrap(openai_recall, recall),
        "vs_untuned": paired_bootstrap(untuned_recall, recall),
        "loss_curve": curve,
    }

report = {
    "smoke": SMOKE,
    "snapshot_version": manifest["snapshot_version"],
    "snapshot_papers_sha256": manifest["snapshot_papers_sha256"],
    "base_model": BASE_MODEL,
    "max_seq_length": MAX_SEQ_LENGTH,
    "hyperparameters": {
        "batch_size": BATCH_SIZE, "mini_batch_size": MINI_BATCH,
        "mini_batch_auto_selected": AUTO_MINI_BATCH and not SMOKE,
        "eval_batch_size": EVAL_BATCH, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "seed": SEED,
        "loss": "CachedMultipleNegativesRankingLoss",
    },
    "corpus": len(corpus_ids),
    "train_pairs": len(train_rows),
    "queries": len(queries),
    "results": results,
}
print(json.dumps(report, ensure_ascii=False, indent=2))
Path("/tmp/colab_finetune_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)

## 읽는 법

**질문 1 — 파인튜닝이 API 모델을 넘는가.** `vs_openai`의 `excludes_zero`를 보세요. `True`이고
`mean_difference`가 양수면 넘은 것입니다. 구간이 0을 포함하면 **넘지 못했거나 측정되지
않았습니다** — 평균이 올라갔다는 것만으로는 부족합니다.

**질문 2 — hard negative가 작동하는가.** `hard`와 `inbatch`의 `vs_untuned`를 비교하세요.
`hard`가 더 크면 그래프에서 뽑은 음성이 기여한 것이고, 선형 어댑터에서의 기각이 **모델 용량
문제**였다는 뜻입니다. 둘이 같거나 `hard`가 낮으면 신호 자체가 약한 것입니다.

**`untuned`가 0.1909을 재현하지 않으면 그 아래 수치는 모두 무의미합니다.** 비교의 전제가
"같은 과제·같은 질의·같은 후보"이기 때문입니다. 차이가 크면 먼저 원인을 찾으세요.

In [ ]:
# 모델과 보고서를 Hugging Face 비공개 저장소에 올립니다(처음에 입력한 토큰, 쓰기 권한 필요).
if not SMOKE:
    from huggingface_hub import HfApi

    api = HfApi(token=token)
    for variant in scored:
        repo = f"{HF_USER}/{HF_REPO_PREFIX}-{variant}"
        api.create_repo(repo, private=True, exist_ok=True)
        api.upload_folder(repo_id=repo, folder_path=f"/tmp/out-{variant}")
        api.upload_file(
            path_or_fileobj="/tmp/colab_finetune_report.json",
            path_in_repo="colab_finetune_report.json", repo_id=repo,
        )
        # 스텝별 손실 CSV도 함께 올립니다. 런타임이 사라져도 곡선이 남습니다.
        loss_csv = Path(f"/tmp/loss-{variant}.csv")
        if loss_csv.exists():
            api.upload_file(
                path_or_fileobj=str(loss_csv),
                path_in_repo=f"loss-{variant}.csv", repo_id=repo,
            )
        print("uploaded", repo)
del token

## 결과를 저장소로 가져오기

`colab_finetune_report.json`을 `reports/embedding_finetune_colab.json`으로 커밋하면 다른
측정들과 같은 자리에 남습니다. 모델 가중치는 커밋하지 않습니다 — HF 비공개 저장소에 두고
보고서만 가져옵니다.